In [2]:
import pandas as pd
import requests
import time
import os
from tqdm import tqdm

OVERPASS_SERVERS = [
    "https://overpass-api.de/api/interpreter",
    "https://lz4.overpass-api.de/api/interpreter",
    "https://overpass.kumi.systems/api/interpreter"
]

headers = {
    "User-Agent": "RentoraAI/1.0 (https://github.com/anisha-1811/Rentora-AI)"
}

CACHE_FILE = "../data/geo_features_cache.csv"
FAILED_FILE = "../data/geo_features_failed.csv"

In [3]:
combined = pd.read_csv("../data/rentora_model_ready.csv", low_memory=False)
print(combined.shape)

unique_locs = combined[["city", "locality", "latitude", "longitude"]].drop_duplicates(subset=["latitude", "longitude"])
print("Unique locations:", len(unique_locs))

if os.path.exists(CACHE_FILE):
    cache = pd.read_csv(CACHE_FILE)
    print("Loaded cache:", len(cache))
else:
    cache = pd.DataFrame(columns=["latitude", "longitude", "near_highway", "near_mall", "near_river", "near_mountain"])

processed = set(zip(cache.latitude, cache.longitude))
remaining = unique_locs[~unique_locs.set_index(["latitude", "longitude"]).index.isin(processed)]
print("Remaining:", len(remaining))

(117443, 10)
Unique locations: 7765
Remaining: 7765


In [4]:
def get_geo_features(lat, lon, radius=2000):
    if pd.isna(lat) or pd.isna(lon):
        return {"near_highway": None, "near_mall": None, "near_river": None, "near_mountain": None}

    query = f"""
[out:json][timeout:25];
(
way["highway"~"trunk|primary|motorway"](around:{radius},{lat},{lon});
node["shop"="mall"](around:{radius},{lat},{lon});
way["natural"="water"](around:{radius},{lat},{lon});
node["natural"="peak"](around:{radius},{lat},{lon});
);
out tags;
"""

    for attempt in range(3):
        for url in OVERPASS_SERVERS:
            try:
                response = requests.post(url, data=query, headers=headers, timeout=120)

                if response.status_code in [429, 500, 502, 503, 504]:
                    wait = (attempt + 1) * 10
                    print(f"{response.status_code} from {url}. Waiting {wait}s...")
                    time.sleep(wait)
                    continue

                response.raise_for_status()
                data = response.json()
                elements = data.get("elements", [])

                return {
                    "near_highway": any(e.get("tags", {}).get("highway") in ["motorway", "primary", "trunk"] for e in elements),
                    "near_mall": any(e.get("tags", {}).get("shop") == "mall" for e in elements),
                    "near_river": any(e.get("tags", {}).get("natural") == "water" for e in elements),
                    "near_mountain": any(e.get("tags", {}).get("natural") == "peak" for e in elements)
                }

            except Exception as e:
                print(f"Error at {url}: {e}")
                continue

    return {"near_highway": None, "near_mall": None, "near_river": None, "near_mountain": None}

In [5]:
priority_cities = ['Delhi', 'Mumbai', 'Pune']
remaining_priority = remaining[remaining['city'].isin(priority_cities)]

sample_size = 500
samples = []
for city in priority_cities:
    city_df = remaining_priority[remaining_priority['city'] == city]
    n = min(len(city_df), sample_size // 3)
    samples.append(city_df.sample(n, random_state=42))

remaining_sample = pd.concat(samples, ignore_index=True)
print("Sample locations to process:", len(remaining_sample))
print(remaining_sample['city'].value_counts())

Sample locations to process: 498
city
Delhi     166
Mumbai    166
Pune      166
Name: count, dtype: int64


In [6]:
import math

def haversine(lat1, lon1, lat2, lon2):
    R = 6371000  # meters
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)
    a = math.sin(dphi/2)**2 + math.cos(phi1)*math.cos(phi2)*math.sin(dlambda/2)**2
    return 2*R*math.asin(math.sqrt(a))

city_centers = {
    'Delhi': (28.7041, 77.1025),
    'Mumbai': (19.0760, 72.8777),
    'Pune': (18.5204, 73.8567)
}

def get_city_features(lat, lon, radius=30000):
    query = f"""
[out:json][timeout:90];
(
way["highway"~"trunk|primary|motorway"](around:{radius},{lat},{lon});
node["shop"="mall"](around:{radius},{lat},{lon});
way["natural"="water"](around:{radius},{lat},{lon});
node["natural"="peak"](around:{radius},{lat},{lon});
);
out center;
"""
    for url in OVERPASS_SERVERS:
        try:
            response = requests.post(url, data=query, headers=headers, timeout=90)
            if response.status_code == 200:
                return response.json().get("elements", [])
        except Exception as e:
            print(f"Error at {url}: {e}")
            continue
    return []

city_features = {}
for city, (lat, lon) in city_centers.items():
    print(f"Fetching bulk features for {city}...")
    city_features[city] = get_city_features(lat, lon)
    print(f"  {city}: {len(city_features[city])} features found")
    time.sleep(3)

Fetching bulk features for Delhi...
  Delhi: 7691 features found
Fetching bulk features for Mumbai...
  Mumbai: 6382 features found
Fetching bulk features for Pune...
  Pune: 4439 features found


In [7]:
#Step 1: Extract coordinates from the bulk results, organized by feature type
def extract_features_by_type(elements):
    highways, malls, rivers, mountains = [], [], [], []
    for e in elements:
        tags = e.get('tags', {})
        # ways use 'center' (lat/lon), nodes have lat/lon directly
        lat = e.get('lat') or e.get('center', {}).get('lat')
        lon = e.get('lon') or e.get('center', {}).get('lon')
        if lat is None or lon is None:
            continue
        if tags.get('highway') in ['motorway', 'primary', 'trunk']:
            highways.append((lat, lon))
        if tags.get('shop') == 'mall':
            malls.append((lat, lon))
        if tags.get('natural') == 'water':
            rivers.append((lat, lon))
        if tags.get('natural') == 'peak':
            mountains.append((lat, lon))
    return highways, malls, rivers, mountains

city_feature_coords = {}
for city, elements in city_features.items():
    city_feature_coords[city] = extract_features_by_type(elements)
    h, m, r, mt = city_feature_coords[city]
    print(f"{city}: {len(h)} highways, {len(m)} malls, {len(r)} rivers, {len(mt)} mountains")

Delhi: 3734 highways, 29 malls, 2318 rivers, 3 mountains
Mumbai: 3967 highways, 23 malls, 1073 rivers, 38 mountains
Pune: 3314 highways, 16 malls, 565 rivers, 18 mountains


In [8]:
#Step 2: Local proximity check function (no network calls — instant)
def check_proximity(lat, lon, city, radius=2000):
    if city not in city_feature_coords:
        return {"near_highway": None, "near_mall": None, "near_river": None, "near_mountain": None}
    highways, malls, rivers, mountains = city_feature_coords[city]
    return {
        "near_highway": any(haversine(lat, lon, h[0], h[1]) <= radius for h in highways),
        "near_mall": any(haversine(lat, lon, m[0], m[1]) <= radius for m in malls),
        "near_river": any(haversine(lat, lon, r[0], r[1]) <= radius for r in rivers),
        "near_mountain": any(haversine(lat, lon, mt[0], mt[1]) <= radius for mt in mountains)
    }

In [9]:
#Step 3: Apply this to every locality in Delhi/Mumbai/Pune — all of them, not just 498
results = []
for _, row in tqdm(remaining_priority.iterrows(), total=len(remaining_priority)):
    features = check_proximity(row.latitude, row.longitude, row.city)
    results.append({"latitude": row.latitude, "longitude": row.longitude, **features})

geo_df = pd.DataFrame(results)
print(geo_df.shape)
geo_df.head()

100%|██████████| 6163/6163 [00:15<00:00, 387.91it/s]

(6163, 6)


,latitude,longitude,near_highway,near_mall,near_river,near_mountain
0,28.529249,77.154134,True,False,True,False
1,28.526191,77.157084,True,False,True,False
2,28.521168,77.202224,True,False,True,False
3,28.589088,77.301508,True,True,True,False
4,28.630597,77.277523,True,False,True,False


In [10]:
#Step 4: extend to the other 5 cities
other_city_centers = {
    'Ahmedabad': (23.0225, 72.5714),
    'Bangalore': (12.9716, 77.5946),
    'Chennai': (13.0827, 80.2707),
    'Hyderabad': (17.3850, 78.4867),
    'Kolkata': (22.5726, 88.3639)
}

for city, (lat, lon) in other_city_centers.items():
    print(f"Fetching bulk features for {city}...")
    city_features[city] = get_city_features(lat, lon)
    print(f"  {city}: {len(city_features[city])} features found")
    time.sleep(3)

# extract coords for these too
for city in other_city_centers:
    city_feature_coords[city] = extract_features_by_type(city_features[city])
    h, m, r, mt = city_feature_coords[city]
    print(f"{city}: {len(h)} highways, {len(m)} malls, {len(r)} rivers, {len(mt)} mountains")

Fetching bulk features for Ahmedabad...
  Ahmedabad: 2667 features found
Fetching bulk features for Bangalore...
  Bangalore: 10831 features found
Fetching bulk features for Chennai...
  Chennai: 5316 features found
Fetching bulk features for Hyderabad...
  Hyderabad: 5766 features found
Fetching bulk features for Kolkata...
  Kolkata: 11749 features found
Ahmedabad: 1856 highways, 6 malls, 377 rivers, 0 mountains
Bangalore: 4695 highways, 14 malls, 5129 rivers, 9 mountains
Chennai: 3188 highways, 8 malls, 1171 rivers, 8 mountains
Hyderabad: 3891 highways, 77 malls, 613 rivers, 7 mountains
Kolkata: 2330 highways, 7 malls, 9186 rivers, 0 mountains


In [11]:
#Step 5 now — this closes out geo-features for all 7,765 locations:
results_all = []
for _, row in tqdm(remaining.iterrows(), total=len(remaining)):
    features = check_proximity(row.latitude, row.longitude, row.city)
    results_all.append({"latitude": row.latitude, "longitude": row.longitude, **features})

geo_df_all = pd.DataFrame(results_all)
print(geo_df_all.shape)
geo_df_all.to_csv("../data/geo_features_all.csv", index=False)
print("Saved!")

100%|██████████| 7765/7765 [00:19<00:00, 390.66it/s]

(7765, 6)
Saved!


In [12]:
# Last step: merge into your main dataset
combined = pd.read_csv("../data/rentora_model_ready.csv", low_memory=False)
geo_features = pd.read_csv("../data/geo_features_all.csv")

final = combined.merge(geo_features, on=["latitude", "longitude"], how="left")
print(final.shape)
print(final[["near_highway", "near_mall", "near_river", "near_mountain"]].isna().sum())

final.to_csv("../data/rentora_final_week1.csv", index=False)
print("Saved!")

(117443, 14)
near_highway     6
near_mall        6
near_river       6
near_mountain    6
dtype: int64
Saved!


In [13]:
# ============================================
# STEP: Load the Week 1 dataset and inspect the small number of rows
# that are missing geo-features after the merge (6 rows out of 117,443)
# Goal: understand WHY they're missing before deciding how to fix them
# ============================================
final = pd.read_csv("../data/rentora_final_week1.csv", low_memory=False)

missing = final[final['near_highway'].isna()]
print(missing[['city', 'locality', 'latitude', 'longitude']])

         city         locality   latitude  longitude
112424  Hisar         DN Nagar  19.123688  72.829529
112539  Hisar  Borivali (West)  19.248964  72.854820
113120  Hisar      Lokhandwala  19.143806  72.826813
113386  Hisar         DN Nagar  19.123688  72.829529
113429  Hisar         DN Nagar  19.118765  72.828758
113766  Hisar          Versova  19.138638  72.810127


In [14]:
# ============================================
# STEP: Fill the 6 missing geo-feature rows with False
# Reasoning: if no highway/mall/river/mountain was found within the search
# radius during extraction, "not near" is a fair default — not a guess,
# since the absence of a match IS the information we have for these rows
# ============================================
geo_cols = ['near_highway', 'near_mall', 'near_river', 'near_mountain']
final[geo_cols] = final[geo_cols].fillna(False)

# confirm no missing values remain in these columns
print(final[geo_cols].isna().sum())

near_highway     0
near_mall        0
near_river       0
near_mountain    0
dtype: int64


In [15]:
# ============================================
# STEP: Save the final, fully clean Week 1 dataset
# This is the official handoff point into Week 2 (model training) —
# no more changes to this file after this point
# ============================================
final.to_csv("../data/rentora_final_week1.csv", index=False)
print("Saved:", final.shape)

Saved: (117443, 14)
